Smart Transaction Validator

In [1]:
def get_valid_input(prompt, input_type=str, valid_options=None, min_val=None, max_val=None):
    """Generic input validator with comprehensive error handling."""
    while True:
        try:
            user_input = input(prompt).strip()
            
            if not user_input:
                print("Error: Input cannot be empty.")
                continue
            
            # Type conversion
            if input_type == int:
                value = int(user_input)
            elif input_type == float:
                value = float(user_input)
            else:
                value = user_input.lower() if valid_options else user_input
            
            # Range validation
            if input_type in (int, float):
                if min_val is not None and value < min_val:
                    print(f"Error: Value must be at least {min_val}.")
                    continue
                if max_val is not None and value > max_val:
                    print(f"Error: Value must be at most {max_val}.")
                    continue
            
            # Option validation
            if valid_options:
                valid_lower = [opt.lower() for opt in valid_options]
                if str(value).lower() not in valid_lower:
                    print(f"Error: Invalid option. Choose from: {', '.join(valid_options)}")
                    continue
                idx = valid_lower.index(str(value).lower())
                return valid_options[idx]
            
            return value
            
        except ValueError:
            print(f"Error: Invalid {input_type.__name__} value.")


def validate_transaction(amount, category, hour, daily_spent_so_far, is_vip=False):
    """
    Validate transaction against fraud detection rules.
    
    Blocking rules (highest priority):
    1. Single transaction > 50,000 (100,000 for VIP)
    2. Daily cumulative > 100,000 (200,000 for VIP)
    
    Flagging rules:
    1. Unusual hours (before 6 AM or after 11 PM)
    2. Category limits exceeded
    
    Returns:
        tuple: (status, reason)
    """
    
    # Dynamic limit configuration using ternary operator
    SINGLE_TX_LIMIT = 100000 if is_vip else 50000
    DAILY_LIMIT = 200000 if is_vip else 100000
    CATEGORY_LIMITS = {
        'food': 10000 if is_vip else 5000,
        'electronics': 60000 if is_vip else 30000,
        'travel': float('inf'),  # No specific limit
        'other': float('inf')
    }
    
    UNUSUAL_HOURS_START = 23  # 11 PM
    UNUSUAL_HOURS_END = 6     # 6 AM
    
    result = {
        'is_vip': is_vip,
        'limits': {
            'single_tx': SINGLE_TX_LIMIT,
            'daily': DAILY_LIMIT,
            'category': CATEGORY_LIMITS.get(category, float('inf'))
        }
    }
    
    # BLOCKING RULES (override everything - checked first)
    
    # Rule 1: Single transaction limit exceeded
    if amount > SINGLE_TX_LIMIT:
        return "BLOCKED", f"Exceeds single transaction limit (Rs.{SINGLE_TX_LIMIT:,})"
    
    # Rule 2: Daily cumulative spending limit exceeded
    projected_daily = daily_spent_so_far + amount
    if projected_daily > DAILY_LIMIT:
        return "BLOCKED", (f"Daily spending limit exceeded "
                          f"(Rs.{projected_daily:,.0f} > Rs.{DAILY_LIMIT:,})")
    
    # FLAGGING RULES (checked if not blocked)
    flags = []
    
    # Rule 3: Unusual hours (before 6 AM or after 11 PM)
    if hour >= UNUSUAL_HOURS_START or hour < UNUSUAL_HOURS_END:
        flags.append(f"Unusual hours ({hour:02d}:00)")
    
    # Rule 4: Category-specific limit exceeded
    cat_limit = CATEGORY_LIMITS.get(category, float('inf'))
    if amount > cat_limit:
        flags.append(f"Exceeds {category} limit (Rs.{amount:,.0f} > Rs.{cat_limit:,})")
    
    # Determine final status
    if flags:
        return "FLAGGED", "; ".join(flags)
    else:
        return "APPROVED", "Transaction within all limits and normal hours"


def format_currency(amount):
    """Format amount in Indian Rupee style."""
    return f"Rs.{amount:,.0f}"


def display_result(amount, category, hour, daily_spent, is_vip, status, reason):
    """Format and display transaction decision."""
    
    print("\n" + "=" * 60)
    print("TRANSACTION VALIDATION REPORT")
    print("=" * 60)
    
    # Transaction Details
    print(f"Amount: {format_currency(amount)}")
    print(f"Category: {category.upper()}")
    print(f"Time: {hour:02d}:00")
    print(f"Daily Spent So Far: {format_currency(daily_spent)}")
    print(f"VIP Status: {'YES' if is_vip else 'NO'}")
    
    # Limits Applied
    multiplier = 2 if is_vip else 1
    print(f"\nLimits Applied (VIP={is_vip}):")
    print(f"  Single Transaction: {format_currency(50000 * multiplier)}")
    print(f"  Daily Cumulative: {format_currency(100000 * multiplier)}")
    if category in ['food', 'electronics']:
        cat_base = 5000 if category == 'food' else 30000
        print(f"  {category.capitalize()}: {format_currency(cat_base * multiplier)}")
    
    print("-" * 60)
    
    # Decision
    status_symbol = "✓" if status == "APPROVED" else "⚠" if status == "FLAGGED" else "✗"
    print(f"{status_symbol} {status}")
    print(f"  Reason: {reason}")
    print("=" * 60)


def main():
    """Main transaction validator interface."""
    
    print("SMART TRANSACTION VALIDATOR")
    print("-" * 40)
    print("Blocking Rules: Amount > 50K | Daily > 100K")
    print("Flagging Rules: Hours (before 6AM/after 11PM) | Category limits")
    print("VIP Mode: All limits doubled")
    print("-" * 40)
    
    # Collect inputs
    amount = get_valid_input(
        "Transaction amount (Rs): ", 
        input_type=float, 
        min_val=0
    )
    
    category = get_valid_input(
        "Category (food/travel/electronics/other): ",
        valid_options=['food', 'travel', 'electronics', 'other']
    )
    
    hour = get_valid_input(
        "Hour of transaction (0-23): ",
        input_type=int,
        min_val=0,
        max_val=23
    )
    
    daily_spent = get_valid_input(
        "Amount already spent today (Rs): ",
        input_type=float,
        min_val=0
    )
    
    vip_input = get_valid_input(
        "VIP customer? (yes/no): ",
        valid_options=['yes', 'no']
    )
    is_vip = (vip_input == 'yes')
    
    # Validate
    status, reason = validate_transaction(
        amount, category, hour, daily_spent, is_vip
    )
    
    # Display
    display_result(amount, category, hour, daily_spent, is_vip, status, reason)


if __name__ == "__main__":
    main()

SMART TRANSACTION VALIDATOR
----------------------------------------
Blocking Rules: Amount > 50K | Daily > 100K
Flagging Rules: Hours (before 6AM/after 11PM) | Category limits
VIP Mode: All limits doubled
----------------------------------------

TRANSACTION VALIDATION REPORT
Amount: Rs.500,000
Category: OTHER
Time: 02:00
Daily Spent So Far: Rs.56,842
VIP Status: YES

Limits Applied (VIP=True):
  Single Transaction: Rs.100,000
  Daily Cumulative: Rs.200,000
------------------------------------------------------------
✗ BLOCKED
  Reason: Exceeds single transaction limit (Rs.100,000)
